In [2]:
import numpy as np  
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix



In [3]:
df = pd.read_csv('MedTech_Masterchart - final (1).csv')


In [4]:
df.shape

(226, 329)

In [5]:
df.head()

,S.no,Age,weight,height,Pulse rate,Pain,Site,staging (TNM),Vata,Pitta,...,K_Q62.2,K_Q63,K_Q64,K_Q65_final,K_Q65,K_Q65.1,K_Q66_final,K_Q66,K_Q66.1,K_Q67
0,1,60,75,168,73,5,Hypopharyngeal,T2N2cM0,25.2,24.4,...,0.5,1,1,1,0,1,0,0,0,1
1,2,61,65,165,91,6,Glottis,T3N1M0,38.2,37.9,...,0.0,0,0,1,0,1,1,0,1,0
2,3,56,70,170,88,2,Glottis,T2N0M0,29.1,32.2,...,0.5,1,1,1,0,1,0,0,0,0
3,4,46,67,185,72,7,Oropharyngeal,T4bN2cM0,29.5,24.0,...,0.5,1,1,1,0,1,0,0,0,1
4,5,63,67,185,85,3,Oropharyngeal,T2N0M0,38.9,21.7,...,0.5,1,1,1,0,1,0,0,0,1


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 226 entries, 0 to 225
Columns: 329 entries, S.no to K_Q67
dtypes: float64(54), int64(273), object(2)
memory usage: 581.0+ KB


In [7]:
print(df.isnull().sum().sort_values(ascending=False).head(30))


P_Q16          141
P_Q19          105
V_Q9            61
P_Q9            61
V_Q18           55
K_Q4            11
V_Q4            11
P_Q13           10
K_Q13           10
P_Q56.3          0
P_Q59            0
P_Q59_final      0
P_Q58            0
P_Q57            0
P_Q56.4          0
P_Q56            0
P_Q56.2          0
P_Q56.1          0
P_Q59.2          0
P_Q56_final      0
P_Q55            0
P_Q54            0
P_Q53.1          0
P_Q53            0
P_Q59.1          0
P_Q62_final      0
P_Q60            0
P_Q61            0
K_Q3             0
K_Q2.1           0
dtype: int64


In [8]:
# We'll handle missing values inside model pipelines to avoid leakage.
print('Missing values (top 30):')
print(df.isnull().sum().sort_values(ascending=False).head(30))


Missing values (top 30):
P_Q16          141
P_Q19          105
V_Q9            61
P_Q9            61
V_Q18           55
K_Q4            11
V_Q4            11
P_Q13           10
K_Q13           10
P_Q56.3          0
P_Q59            0
P_Q59_final      0
P_Q58            0
P_Q57            0
P_Q56.4          0
P_Q56            0
P_Q56.2          0
P_Q56.1          0
P_Q59.2          0
P_Q56_final      0
P_Q55            0
P_Q54            0
P_Q53.1          0
P_Q53            0
P_Q59.1          0
P_Q62_final      0
P_Q60            0
P_Q61            0
K_Q3             0
K_Q2.1           0
dtype: int64


In [9]:
# Get all column names
cols = df.columns

# Columns to keep
keep_cols = []

for col in cols:
    # If it's a "_final" column → always keep
    if col.endswith('_final'):
        keep_cols.append(col)
    else:
        # Check if corresponding _final exists
        base = col.split('.')[0]  # handles K_Q65.1
        final_col = base + '_final'
        
        if final_col not in cols:
            keep_cols.append(col)

# Keep only selected columns
df = df[keep_cols]

In [10]:
df.head()

,S.no,Age,weight,height,Pulse rate,Pain,Site,staging (TNM),Vata,Pitta,...,K_Q58,K_Q59_final,K_Q60,K_Q61,K_Q62_final,K_Q63,K_Q64,K_Q65_final,K_Q66_final,K_Q67
0,1,60,75,168,73,5,Hypopharyngeal,T2N2cM0,25.2,24.4,...,0.5,1,1,0.5,1.0,1,1,1,0,1
1,2,61,65,165,91,6,Glottis,T3N1M0,38.2,37.9,...,0.5,1,0,0.0,0.0,0,0,1,1,0
2,3,56,70,170,88,2,Glottis,T2N0M0,29.1,32.2,...,0.5,1,1,0.5,1.0,1,1,1,0,0
3,4,46,67,185,72,7,Oropharyngeal,T4bN2cM0,29.5,24.0,...,0.5,1,1,0.5,1.0,1,1,1,0,1
4,5,63,67,185,85,3,Oropharyngeal,T2N0M0,38.9,21.7,...,0.5,1,1,0.5,1.0,1,1,1,0,1


In [11]:
df.shape

(226, 251)

In [12]:
print(df.columns.tolist())

['S.no', 'Age', 'weight', 'height', 'Pulse rate', 'Pain', 'Site', 'staging (TNM)', 'Vata', 'Pitta', 'Kapha', 'SF 36', 'Value_Q1', 'Value_Q2', 'Value_Q3', 'Value_Q4', 'Value_Q5', 'Value_Q6', 'Value_Q7', 'Value_Q8', 'Value_Q9', 'Value_Q10', 'Value_Q11', 'Value_Q12', 'Value_Q13', 'Value_Q14', 'Value_Q15', 'Value_Q16', 'Value_Q17', 'Value_Q18', 'Value_Q19', 'Value_Q20', 'Value_Q21', 'Value_Q22', 'Value_Q23', 'Value_Q24', 'Value_Q25', 'Value_Q26', 'Value_Q27', 'Value_Q28', 'Value_Q29', 'Value_Q30', 'Value_Q31', 'Value_Q32', 'Value_Q33', 'Value_Q34', 'Value_Q35', 'Value_Q36', 'V_Q1', 'V_Q2_final', 'V_Q3', 'V_Q4', 'V_Q5', 'V_Q6', 'V_Q7', 'V_Q8', 'V_Q9', 'V_Q10', 'V_Q11', 'V_Q12', 'V_Q13', 'V_Q14', 'V_Q15', 'V_Q16', 'V_Q17_final', 'V_Q18', 'V_Q19', 'V_Q20', 'V_Q21_final', 'V_Q22', 'V_Q23', 'V_Q24', 'V_Q25', 'V_Q26', 'V_Q27', 'V_Q28', 'V_Q29', 'V_Q30', 'V_Q31', 'V_Q32', 'V_Q33', 'V_Q34', 'V_Q35', 'V_Q36', 'V_Q37', 'V_Q38', 'V_Q39', 'V_Q40', 'V_Q41', 'V_Q42', 'V_Q43', 'V_Q44', 'V_Q45', 'V_Q46', 

In [13]:
for col in df.columns:
    print(f'{col}: {df[col].dtypes}')

S.no: int64
Age: int64
weight: int64
height: int64
Pulse rate: int64
Pain: int64
Site: object
staging (TNM): object
Vata: float64
Pitta: float64
Kapha: float64
SF 36: int64
Value_Q1: int64
Value_Q2: int64
Value_Q3: int64
Value_Q4: int64
Value_Q5: int64
Value_Q6: int64
Value_Q7: int64
Value_Q8: int64
Value_Q9: int64
Value_Q10: int64
Value_Q11: int64
Value_Q12: int64
Value_Q13: int64
Value_Q14: int64
Value_Q15: int64
Value_Q16: int64
Value_Q17: int64
Value_Q18: int64
Value_Q19: int64
Value_Q20: int64
Value_Q21: float64
Value_Q22: int64
Value_Q23: float64
Value_Q24: float64
Value_Q25: float64
Value_Q26: float64
Value_Q27: float64
Value_Q28: float64
Value_Q29: float64
Value_Q30: float64
Value_Q31: float64
Value_Q32: int64
Value_Q33: int64
Value_Q34: int64
Value_Q35: int64
Value_Q36: int64
V_Q1: int64
V_Q2_final: int64
V_Q3: int64
V_Q4: float64
V_Q5: int64
V_Q6: int64
V_Q7: int64
V_Q8: int64
V_Q9: float64
V_Q10: int64
V_Q11: int64
V_Q12: int64
V_Q13: int64
V_Q14: int64
V_Q15: int64
V_Q16: i

In [14]:
df['Site'].value_counts()

Site
Oral cavity                 62
Supraglottic                39
Glottic                     33
Hypopharyngeal              30
Oropharyngeal               23
Sinonasal carcinoma         10
Nasopharyngeal Carcinoma     8
Maxillary sinus              5
Sinonasal Carcinoma          4
Glottis                      3
Esophagus                    2
Esophageal                   1
Sinonasal chondrosarcoma     1
Oropharynx                   1
sinonasal carcinoma          1
Sinonasal Sarcoma            1
Nasopharyngeal carcinoma     1
Transglottic                 1
Name: count, dtype: int64

In [19]:
import re
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

def split_tnm_into_t_n(raw, keep_suffixes=False):
    """
    Extracts T and N from a raw TNM string.

    Examples:
        T2aN1bM0  ->  T2, N1   (default)
        T4bN2cM1  ->  T4, N2   (default)
        TisN0M0   ->  Tis, N0

    If keep_suffixes=True:
        T2a -> T2a
        N1b -> N1b
    """
    if pd.isna(raw):
        return np.nan, np.nan

    s = str(raw).upper().strip()
    s = re.sub(r'[\s\-_\/]+', '', s)
    s = re.sub(r'[^TNM0-9ABCIS]', '', s)

    t_match = re.search(r'T(IS|[0-4][ABC]?)', s)
    n_match = re.search(r'N([0-3][ABC]?)', s)

    if not (t_match and n_match):
        return np.nan, np.nan

    t_tok = t_match.group(1)
    n_tok = n_match.group(1)

    # T label
    if t_tok == "IS":
        t_label = "Tis"
    else:
        t_base = t_tok[0]
        t_suffix = t_tok[1:] if len(t_tok) > 1 else ""
        t_label = f"T{t_base}{t_suffix.lower()}" if keep_suffixes and t_suffix else f"T{t_base}"

    # N label
    n_base = n_tok[0]
    n_suffix = n_tok[1:] if len(n_tok) > 1 else ""
    n_label = f"N{n_base}{n_suffix.lower()}" if keep_suffixes and n_suffix else f"N{n_base}"

    return t_label, n_label


def add_tn_columns(df, stage_col="staging (TNM)", keep_suffixes=False):
    """
    Adds:
      - T_stage : encoded T label
      - N_stage : encoded N label

    Also returns the fitted encoders.
    """
    df = df.copy()

    if stage_col not in df.columns:
        raise ValueError(f"Column '{stage_col}' not found")

    parsed = df[stage_col].apply(lambda x: split_tnm_into_t_n(x, keep_suffixes=keep_suffixes))
    parsed_df = pd.DataFrame(parsed.tolist(), columns=["T_label", "N_label"], index=df.index)

    df = pd.concat([df, parsed_df], axis=1)

    # keep only rows where both T and N were successfully parsed
    mask = df["T_label"].notna() & df["N_label"].notna()
    df = df.loc[mask].copy()

    t_le = LabelEncoder()
    n_le = LabelEncoder()

    df["T_stage"] = t_le.fit_transform(df["T_label"].astype(str))
    df["N_stage"] = n_le.fit_transform(df["N_label"].astype(str))

    # remove helper string columns
    df.drop(columns=["T_label", "N_label"], inplace=True)

    return df, t_le, n_le


# Usage
df, t_le, n_le = add_tn_columns(df, stage_col="staging (TNM)", keep_suffixes=False)

print("T classes:", list(t_le.classes_))
print("N classes:", list(n_le.classes_))
print(df[["staging (TNM)", "T_stage", "N_stage"]].head())

T classes: ['T1', 'T2', 'T3', 'T4']
N classes: ['N0', 'N1', 'N2', 'N3']
  staging (TNM)  T_stage  N_stage
0       T2N2cM0        1        2
1        T3N1M0        2        1
2        T2N0M0        1        0
3      T4bN2cM0        3        2
4        T2N0M0        1        0


In [20]:
df


,S.no,Age,weight,height,Pulse rate,Pain,Site,staging (TNM),Vata,Pitta,...,K_Q60,K_Q61,K_Q62_final,K_Q63,K_Q64,K_Q65_final,K_Q66_final,K_Q67,T_stage,N_stage
0,1,60,75,168,73,5,Hypopharyngeal,T2N2cM0,25.2,24.4,...,1,0.5,1.0,1,1,1,0,1,1,2
1,2,61,65,165,91,6,Glottis,T3N1M0,38.2,37.9,...,0,0.0,0.0,0,0,1,1,0,2,1
2,3,56,70,170,88,2,Glottis,T2N0M0,29.1,32.2,...,1,0.5,1.0,1,1,1,0,0,1,0
3,4,46,67,185,72,7,Oropharyngeal,T4bN2cM0,29.5,24.0,...,1,0.5,1.0,1,1,1,0,1,3,2
4,5,63,67,185,85,3,Oropharyngeal,T2N0M0,38.9,21.7,...,1,0.5,1.0,1,1,1,0,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
221,222,61,75,176,76,3,Hypopharyngeal,T3N0M0,29.5,24.0,...,1,0.5,1.0,1,1,1,0,1,2,0
222,223,51,69,175,71,3,Oral cavity,T4aN2bM0,29.3,31.0,...,1,0.5,1.0,1,1,1,0,0,3,2
223,224,70,61,166,77,5,Glottic,T3N0M0,38.9,21.7,...,1,0.5,1.0,1,1,1,0,1,2,0
224,225,75,52,176,74,3,Oropharyngeal,T3N1M0,36.6,23.9,...,1,0.5,1.0,1,1,1,0,1,2,1


In [22]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# -----------------------------
# 1. Define numeric columns
# -----------------------------
numeric_cols = ["Pain", "weight", "height", "Pulse rate"]

# Ensure numeric conversion
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors="coerce")

# -----------------------------
# 2. Filter valid rows
# -----------------------------
# Now we use T_stage and N_stage instead of stage_clean
required_cols = ["T_stage", "N_stage"] + numeric_cols

mask = pd.Series(True, index=df.index)

for col in required_cols:
    mask &= df[col].notna()

df = df.loc[mask].copy()

print("After filtering:", df.shape)

# -----------------------------
# 3. Min-Max scaling (replace original columns)
# -----------------------------
scaler = MinMaxScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

# -----------------------------
# Final dataset ready
# -----------------------------
print("\nFinal shape:", df.shape)
print(df[["T_stage", "N_stage"] + numeric_cols].head())

After filtering: (226, 253)

Final shape: (226, 253)
   T_stage  N_stage   Pain    weight    height  Pulse rate
0        1        2  0.500  0.681818  0.387097    0.343750
1        2        1  0.625  0.454545  0.290323    0.625000
2        1        0  0.125  0.568182  0.451613    0.578125
3        3        2  0.750  0.500000  0.935484    0.328125
4        1        0  0.250  0.500000  0.935484    0.531250


In [23]:
from sklearn.preprocessing import OneHotEncoder
import pandas as pd

# ensure df is DataFrame
print(type(df))

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

encoded = encoder.fit_transform(df[["Site"]])

encoded_df = pd.DataFrame(
    encoded,
    columns=encoder.get_feature_names_out(["Site"])
)

df = pd.concat([df, encoded_df], axis=1)
df.drop("Site", axis=1, inplace=True)

<class 'pandas.core.frame.DataFrame'>


In [24]:
df

,S.no,Age,weight,height,Pulse rate,Pain,staging (TNM),Vata,Pitta,Kapha,...,Site_Oral cavity,Site_Oropharyngeal,Site_Oropharynx,Site_Sinonasal Carcinoma,Site_Sinonasal Sarcoma,Site_Sinonasal carcinoma,Site_Sinonasal chondrosarcoma,Site_Supraglottic,Site_Transglottic,Site_sinonasal carcinoma
0,1,60,0.681818,0.387097,0.343750,0.500,T2N2cM0,25.2,24.4,50.4,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,61,0.454545,0.290323,0.625000,0.625,T3N1M0,38.2,37.9,23.9,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,56,0.568182,0.451613,0.578125,0.125,T2N0M0,29.1,32.2,38.8,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,46,0.500000,0.935484,0.328125,0.750,T4bN2cM0,29.5,24.0,46.5,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5,63,0.500000,0.935484,0.531250,0.250,T2N0M0,38.9,21.7,39.3,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
221,222,61,0.681818,0.645161,0.390625,0.250,T3N0M0,29.5,24.0,46.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
222,223,51,0.545455,0.612903,0.312500,0.250,T4aN2bM0,29.3,31.0,39.7,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
223,224,70,0.363636,0.322581,0.406250,0.500,T3N0M0,38.9,21.7,39.3,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
224,225,75,0.159091,0.645161,0.359375,0.250,T3N1M0,36.6,23.9,39.5,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [27]:
import pandas as pd
import numpy as np
import re

def prepare_dataset(df, mode="value"):
    """
    mode options:
    - "value" → Value_Q1 to Value_Q36
    - "vpk"   → V_Q*, P_Q*, K_Q*
    - "all"   → Value_Q* + V_Q* + P_Q* + K_Q*

    Returns:
      X        : feature matrix
      y_T      : T_stage
      y_N      : N_stage
      y_pain   : Pain
    """

    df = df.copy()

    # -----------------------------
    # 1. Clean column names
    # -----------------------------
    df.columns = df.columns.str.strip()
    df.columns = [re.sub(r'\.\d+$', '', col) for col in df.columns]
    df.columns = [col.replace('-', '_') for col in df.columns]
    df = df.loc[:, ~df.columns.duplicated()]

    # -----------------------------
    # 2. Targets
    # -----------------------------
    required_targets = ["T_stage", "N_stage", "Pain"]

    for col in required_targets:
        if col not in df.columns:
            raise ValueError(f"Target column '{col}' not found")

    y_T = pd.to_numeric(df["T_stage"], errors="coerce")
    y_N = pd.to_numeric(df["N_stage"], errors="coerce")
    y_pain = pd.to_numeric(df["Pain"], errors="coerce")

    # Keep only valid rows
    mask = y_T.notna() & y_N.notna() & y_pain.notna()
    df = df.loc[mask].copy()

    y_T = y_T.loc[mask].astype(int)
    y_N = y_N.loc[mask].astype(int)
    y_pain = y_pain.loc[mask].astype(float)

    # -----------------------------
    # 3. Select feature columns
    # -----------------------------
    if mode == "value":
        feature_cols = [col for col in df.columns if col.startswith("Value_Q")]

    elif mode == "vpk":
        feature_cols = [
            col for col in df.columns
            if col.startswith("V_Q") or col.startswith("P_Q") or col.startswith("K_Q")
        ]

    elif mode == "all":
        feature_cols = [
            col for col in df.columns
            if col.startswith("Value_Q")
            or col.startswith("V_Q")
            or col.startswith("P_Q")
            or col.startswith("K_Q")
        ]

    else:
        raise ValueError("Invalid mode. Choose from: value, vpk, all")

    # -----------------------------
    # 4. Add Site encoded columns
    # -----------------------------
    site_cols = [col for col in df.columns if col.startswith("Site_")]

    # -----------------------------
    # 5. Final feature matrix
    # -----------------------------
    final_cols = feature_cols + site_cols

    X = (
        df[final_cols]
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
    )

    return (
        np.nan_to_num(np.asarray(X, dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0),
        np.asarray(y_T),
        np.asarray(y_N),
        np.asarray(y_pain),
    )


# -----------------------------
# Example usage
# -----------------------------
X1, y_T1, y_N1, y_pain1 = prepare_dataset(df, mode="value")
X2, y_T2, y_N2, y_pain2 = prepare_dataset(df, mode="vpk")
X3, y_T3, y_N3, y_pain3 = prepare_dataset(df, mode="all")

print(X1.shape, X2.shape, X3.shape)
print("T classes:", len(np.unique(y_T1)))
print("N classes:", len(np.unique(y_N1)))
print("Pain range:", y_pain1.min(), y_pain1.max())

(226, 54) (226, 220) (226, 256)
T classes: 4
N classes: 4
Pain range: 0.0 1.0


In [29]:
y_N1

array([2, 1, 0, 2, 0, 1, 0, 1, 1, 1, 2, 3, 1, 0, 1, 2, 0, 1, 0, 3, 1, 1,
       3, 2, 0, 3, 1, 1, 1, 2, 1, 0, 0, 0, 0, 1, 2, 1, 1, 0, 2, 2, 0, 1,
       0, 1, 2, 1, 0, 2, 0, 2, 1, 1, 1, 2, 0, 0, 2, 1, 1, 1, 1, 1, 0, 0,
       0, 1, 0, 3, 1, 1, 1, 0, 0, 1, 1, 1, 2, 1, 2, 1, 2, 0, 1, 1, 2, 1,
       2, 2, 1, 1, 1, 0, 1, 0, 0, 1, 3, 3, 1, 3, 2, 0, 0, 1, 1, 1, 1, 2,
       1, 0, 2, 1, 3, 2, 1, 2, 1, 2, 1, 1, 2, 0, 1, 0, 1, 1, 1, 1, 0, 1,
       3, 1, 1, 1, 1, 0, 2, 1, 1, 1, 1, 2, 1, 0, 0, 1, 1, 0, 2, 3, 0, 2,
       1, 1, 2, 2, 0, 2, 1, 1, 0, 2, 0, 1, 0, 1, 1, 0, 1, 0, 2, 0, 1, 1,
       2, 1, 2, 1, 1, 2, 0, 1, 2, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 3, 0, 3,
       2, 1, 1, 1, 2, 1, 0, 1, 1, 1, 0, 0, 3, 0, 2, 1, 1, 2, 2, 0, 2, 1,
       2, 0, 2, 0, 1, 0])

In [30]:
import numpy as np
from collections import Counter

def fix_single_sample_classes(X, y_T, y_N, y_pain, min_count=3):
    """
    Ensures each (T, N) class combination has at least `min_count` samples
    by duplicating rows.

    Parameters:
        X       : feature matrix (numpy array)
        y_T     : T_stage labels
        y_N     : N_stage labels
        y_pain  : regression target
        min_count : minimum samples required per class

    Returns:
        X_new, y_T_new, y_N_new, y_pain_new
    """

    X = np.asarray(X)
    y_T = np.asarray(y_T)
    y_N = np.asarray(y_N)
    y_pain = np.asarray(y_pain)

    # 🔥 Combine T and N into joint class
    y_joint = np.array([f"{t}_{n}" for t, n in zip(y_T, y_N)])

    class_counts = Counter(y_joint)

    X_list = [X]
    yT_list = [y_T]
    yN_list = [y_N]
    ypain_list = [y_pain]

    for cls, count in class_counts.items():
        if count < min_count:
            needed = min_count - count

            # indices of this joint class
            idx = np.where(y_joint == cls)[0]

            # duplicate
            dup_idx = np.random.choice(idx, size=needed, replace=True)

            X_list.append(X[dup_idx])
            yT_list.append(y_T[dup_idx])
            yN_list.append(y_N[dup_idx])
            ypain_list.append(y_pain[dup_idx])

    # combine
    X_new = np.vstack(X_list)
    y_T_new = np.concatenate(yT_list)
    y_N_new = np.concatenate(yN_list)
    y_pain_new = np.concatenate(ypain_list)

    return X_new, y_T_new, y_N_new, y_pain_new

In [33]:
import numpy as np
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


class MultiTaskNet(nn.Module):
    def __init__(
        self,
        input_dim,
        n_T_classes,
        n_N_classes,
        hidden_dims=(64, 32, 16),
        dropout=0.10
    ):
        super().__init__()

        h1, h2, h3 = hidden_dims

        # -----------------------------
        # Shared backbone
        # -----------------------------
        self.shared = nn.Sequential(
            nn.Linear(input_dim, h1),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(h1, h2),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(h2, h3),
            nn.ReLU()
        )

        # -----------------------------
        # T classification head
        # -----------------------------
        self.T_head = nn.Sequential(
            nn.Linear(h3, h3),
            nn.ReLU(),
            nn.Linear(h3, n_T_classes)
        )

        # -----------------------------
        # N classification head
        # -----------------------------
        self.N_head = nn.Sequential(
            nn.Linear(h3, h3),
            nn.ReLU(),
            nn.Linear(h3, n_N_classes)
        )

        # -----------------------------
        # Pain regression head
        # -----------------------------
        self.pain_head = nn.Sequential(
            nn.Linear(h3, h3),
            nn.ReLU(),
            nn.Linear(h3, 1)
        )

    def forward(self, x):
        shared = self.shared(x)

        T_logits = self.T_head(shared)
        N_logits = self.N_head(shared)
        pain_pred = self.pain_head(shared)

        return T_logits, N_logits, pain_pred


# ---------------------------------------------------
# Build function with 3 architectures
# ---------------------------------------------------
def build_model(input_dim, n_T_classes, n_N_classes, arch="big"):
    """
    arch options:
    - "small" → 32-16-8
    - "big"   → 64-32-16
    - "ultra" → 128-64-32
    """

    if arch == "small":
        hidden_dims = (32, 16, 8)
        dropout = 0.05

    elif arch == "big":
        hidden_dims = (64, 32, 16)
        dropout = 0.10

    elif arch == "ultra":
        hidden_dims = (128, 64, 32)
        dropout = 0.15  # slightly higher to control overfitting

    else:
        raise ValueError("arch must be 'small', 'big', or 'ultra'")

    model = MultiTaskNet(
        input_dim=input_dim,
        n_T_classes=n_T_classes,
        n_N_classes=n_N_classes,
        hidden_dims=hidden_dims,
        dropout=dropout
    )

    return model.to(device)

Using device: cpu


In [34]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    explained_variance_score,
    confusion_matrix
)
import numpy as np


# ---------------------------------------------------
# Regression metrics (UNCHANGED)
# ---------------------------------------------------
def regression_debug_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    mse = mean_squared_error(y_true, y_pred)
    rmse = float(np.sqrt(mse))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    evs = explained_variance_score(y_true, y_pred)

    pred_var = float(np.var(y_pred))
    true_var = float(np.var(y_true))
    resid = y_true - y_pred
    resid_std = float(np.std(resid))

    return {
        "MAE": float(mae),
        "MSE": float(mse),
        "RMSE": rmse,
        "R2": float(r2),
        "ExplainedVariance": float(evs),
        "PredVariance": pred_var,
        "TrueVariance": true_var,
        "ResidualStd": resid_std,
    }


# ---------------------------------------------------
# Classification metrics (for ONE head)
# ---------------------------------------------------
def classification_debug_metrics(y_true, y_pred, y_prob=None):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    acc = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")

    out = {
        "Accuracy": float(acc),
        "BalancedAccuracy": float(bal_acc),
        "MacroF1": float(macro_f1),
        "ConfusionMatrix": confusion_matrix(y_true, y_pred)
    }

    if y_prob is not None:
        y_prob = np.asarray(y_prob)
        max_conf = np.max(y_prob, axis=1)
        entropy = -np.sum(y_prob * np.log(np.clip(y_prob, 1e-12, 1.0)), axis=1)

        out["MeanSoftmaxConfidence"] = float(np.mean(max_conf))
        out["MedianSoftmaxConfidence"] = float(np.median(max_conf))
        out["MeanEntropy"] = float(np.mean(entropy))

    return out


# ---------------------------------------------------
# Print block for BOTH T and N
# ---------------------------------------------------
def print_metrics_block(title, T_metrics, N_metrics, reg_metrics):
    print(f"\n===== {title} =====")

    # -----------------------------
    # T metrics
    # -----------------------------
    print("\n--- T Classification ---")
    print(
        f"Acc: {T_metrics['Accuracy']:.4f} | "
        f"BalAcc: {T_metrics['BalancedAccuracy']:.4f} | "
        f"MacroF1: {T_metrics['MacroF1']:.4f}"
    )

    if "MeanSoftmaxConfidence" in T_metrics:
        print(
            f"Confidence | Mean: {T_metrics['MeanSoftmaxConfidence']:.4f} | "
            f"Median: {T_metrics['MedianSoftmaxConfidence']:.4f} | "
            f"Entropy: {T_metrics['MeanEntropy']:.4f}"
        )

    print("Confusion matrix (T):\n", T_metrics["ConfusionMatrix"])

    # -----------------------------
    # N metrics
    # -----------------------------
    print("\n--- N Classification ---")
    print(
        f"Acc: {N_metrics['Accuracy']:.4f} | "
        f"BalAcc: {N_metrics['BalancedAccuracy']:.4f} | "
        f"MacroF1: {N_metrics['MacroF1']:.4f}"
    )

    if "MeanSoftmaxConfidence" in N_metrics:
        print(
            f"Confidence | Mean: {N_metrics['MeanSoftmaxConfidence']:.4f} | "
            f"Median: {N_metrics['MedianSoftmaxConfidence']:.4f} | "
            f"Entropy: {N_metrics['MeanEntropy']:.4f}"
        )

    print("Confusion matrix (N):\n", N_metrics["ConfusionMatrix"])

    # -----------------------------
    # Pain regression
    # -----------------------------
    print("\n--- Pain Regression ---")
    print(
        f"MAE: {reg_metrics['MAE']:.4f} | "
        f"RMSE: {reg_metrics['RMSE']:.4f} | "
        f"MSE: {reg_metrics['MSE']:.4f} | "
        f"R2: {reg_metrics['R2']:.4f} | "
        f"EVS: {reg_metrics['ExplainedVariance']:.4f}"
    )

    print(
        f"Variance | Pred: {reg_metrics['PredVariance']:.4f} | "
        f"True: {reg_metrics['TrueVariance']:.4f} | "
        f"ResidualStd: {reg_metrics['ResidualStd']:.4f}"
    )

In [35]:
from collections import Counter
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

def run_model(
    X,
    y_T,
    y_N,
    y_pain,
    dataset_name="dataset",
    arch="big",
    min_count=3,
    test_size=0.20,
    random_state=42,
    batch_size=16,
    epochs=120,
    lr=3e-3,
    weight_decay=1e-5,
    T_loss_w=1.0,
    N_loss_w=1.0,
    pain_loss_w=0.5
):

    # ---------------------------------------------------
    # 1) Convert to numpy
    # ---------------------------------------------------
    X = np.asarray(X, dtype=np.float32)
    y_T = np.asarray(y_T, dtype=np.int64).reshape(-1)
    y_N = np.asarray(y_N, dtype=np.int64).reshape(-1)
    y_pain = np.asarray(y_pain, dtype=np.float32).reshape(-1, 1)

    print(f"\n--- {dataset_name.upper()} ---")
    print("Original shape:", X.shape)
    print("T class counts:", dict(Counter(y_T)))
    print("N class counts:", dict(Counter(y_N)))

    # ---------------------------------------------------
    # 2) Train/test split (FIRST)
    # ---------------------------------------------------
    X_train, X_test, yT_train, yT_test, yN_train, yN_test, ypain_train, ypain_test = train_test_split(
        X,
        y_T,
        y_N,
        y_pain,
        test_size=test_size,
        random_state=random_state,
        stratify=y_T   # stratify on T (more stable than joint)
    )

    # ---------------------------------------------------
    # 3) Apply replication ONLY on train
    # ---------------------------------------------------
    X_train, yT_train, yN_train, ypain_train = fix_single_sample_classes(
        X_train, yT_train, yN_train, ypain_train, min_count=min_count
    )

    print("Train shape (after balancing):", X_train.shape)
    print("Test shape:", X_test.shape)

    # ---------------------------------------------------
    # 4) DataLoaders
    # ---------------------------------------------------
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    yT_train_t = torch.tensor(yT_train, dtype=torch.long)
    yN_train_t = torch.tensor(yN_train, dtype=torch.long)
    ypain_train_t = torch.tensor(ypain_train, dtype=torch.float32)

    X_test_t = torch.tensor(X_test, dtype=torch.float32)
    yT_test_t = torch.tensor(yT_test, dtype=torch.long)
    yN_test_t = torch.tensor(yN_test, dtype=torch.long)
    ypain_test_t = torch.tensor(ypain_test, dtype=torch.float32)

    train_ds = TensorDataset(X_train_t, yT_train_t, yN_train_t, ypain_train_t)
    test_ds = TensorDataset(X_test_t, yT_test_t, yN_test_t, ypain_test_t)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

    # ---------------------------------------------------
    # 5) Model
    # ---------------------------------------------------
    n_T_classes = len(np.unique(y_T))
    n_N_classes = len(np.unique(y_N))

    model = build_model(
        input_dim=X.shape[1],
        n_T_classes=n_T_classes,
        n_N_classes=n_N_classes,
        arch=arch
    )

    T_criterion = nn.CrossEntropyLoss()
    N_criterion = nn.CrossEntropyLoss()
    pain_criterion = nn.MSELoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    total_steps = max(1, epochs * len(train_loader))
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=lr,
        total_steps=total_steps,
        pct_start=0.25,
        anneal_strategy="cos",
        div_factor=10.0,
        final_div_factor=100.0
    )

    use_amp = device.type == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    # ---------------------------------------------------
    # 6) Training loop
    # ---------------------------------------------------
    for epoch in range(1, epochs + 1):
        model.train()

        total_loss = 0
        T_correct = 0
        N_correct = 0
        total = 0

        for xb, yTb, yNb, ypb in train_loader:
            xb = xb.to(device)
            yTb = yTb.to(device)
            yNb = yNb.to(device)
            ypb = ypb.to(device).squeeze(-1)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=use_amp):
                T_logits, N_logits, pain_pred = model(xb)
                pain_pred = pain_pred.squeeze(-1)

                loss_T = T_criterion(T_logits, yTb)
                loss_N = N_criterion(N_logits, yNb)
                loss_pain = pain_criterion(pain_pred, ypb)

                loss = (
                    T_loss_w * loss_T +
                    N_loss_w * loss_N +
                    pain_loss_w * loss_pain
                )

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            total_loss += loss.item() * xb.size(0)

            T_pred = torch.argmax(T_logits, dim=1)
            N_pred = torch.argmax(N_logits, dim=1)

            T_correct += (T_pred == yTb).sum().item()
            N_correct += (N_pred == yNb).sum().item()
            total += yTb.size(0)

        print(
            f"Epoch {epoch:03d} | "
            f"Loss: {total_loss/total:.4f} | "
            f"T_acc: {T_correct/total:.4f} | "
            f"N_acc: {N_correct/total:.4f}"
        )

    # ---------------------------------------------------
    # 7) Evaluation
    # ---------------------------------------------------
    model.eval()

    all_T_true, all_T_pred, all_T_prob = [], [], []
    all_N_true, all_N_pred, all_N_prob = [], [], []
    all_pain_true, all_pain_pred = [], []

    with torch.no_grad():
        for xb, yTb, yNb, ypb in test_loader:
            xb = xb.to(device)

            T_logits, N_logits, pain_pred = model(xb)

            T_prob = torch.softmax(T_logits, dim=1)
            N_prob = torch.softmax(N_logits, dim=1)

            all_T_true.extend(yTb.numpy())
            all_T_pred.extend(torch.argmax(T_prob, dim=1).cpu().numpy())
            all_T_prob.extend(T_prob.cpu().numpy())

            all_N_true.extend(yNb.numpy())
            all_N_pred.extend(torch.argmax(N_prob, dim=1).cpu().numpy())
            all_N_prob.extend(N_prob.cpu().numpy())

            all_pain_true.extend(ypb.numpy())
            all_pain_pred.extend(pain_pred.squeeze(-1).cpu().numpy())

    T_metrics = classification_debug_metrics(
        all_T_true, all_T_pred, all_T_prob
    )
    N_metrics = classification_debug_metrics(
        all_N_true, all_N_pred, all_N_prob
    )
    reg_metrics = regression_debug_metrics(
        all_pain_true, all_pain_pred
    )

    print_metrics_block(dataset_name, T_metrics, N_metrics, reg_metrics)

    return {
        "model": model,
        "T_metrics": T_metrics,
        "N_metrics": N_metrics,
        "regression": reg_metrics
    }

In [36]:

# -----------------------------
# VALUE MODE
# -----------------------------
results_value_small = run_model(
    X1, y_T1, y_N1, y_pain1,
    dataset_name="value_small",
    arch="small"
)

results_value_big = run_model(
    X1, y_T1, y_N1, y_pain1,
    dataset_name="value_big",
    arch="big"
)

results_value_ultra = run_model(
    X1, y_T1, y_N1, y_pain1,
    dataset_name="value_ultra",
    arch="ultra"
)


# -----------------------------
# VPK MODE
# -----------------------------
results_vpk_small = run_model(
    X2, y_T2, y_N2, y_pain2,
    dataset_name="vpk_small",
    arch="small"
)

results_vpk_big = run_model(
    X2, y_T2, y_N2, y_pain2,
    dataset_name="vpk_big",
    arch="big"
)

results_vpk_ultra = run_model(
    X2, y_T2, y_N2, y_pain2,
    dataset_name="vpk_ultra",
    arch="ultra"
)


# -----------------------------
# ALL MODE
# -----------------------------
results_all_small = run_model(
    X3, y_T3, y_N3, y_pain3,
    dataset_name="all_small",
    arch="small"
)

results_all_big = run_model(
    X3, y_T3, y_N3, y_pain3,
    dataset_name="all_big",
    arch="big"
)

results_all_ultra = run_model(
    X3, y_T3, y_N3, y_pain3,
    dataset_name="all_ultra",
    arch="ultra"
)


--- VALUE_SMALL ---
Original shape: (226, 54)
T class counts: {1: 35, 2: 133, 3: 53, 0: 5}
N class counts: {2: 48, 1: 106, 0: 58, 3: 14}
Train shape (after balancing): (182, 54)
Test shape: (46, 54)


/var/folders/ff/flzt5myn3x55cy_nz85y_ynr0000gn/T/ipykernel_18667/2434738068.py:114: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/var/folders/ff/flzt5myn3x55cy_nz85y_ynr0000gn/T/ipykernel_18667/2434738068.py:135: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


Epoch 001 | Loss: 2.7791 | T_acc: 0.5824 | N_acc: 0.2582
Epoch 002 | Loss: 2.7588 | T_acc: 0.5824 | N_acc: 0.2582
Epoch 003 | Loss: 2.7416 | T_acc: 0.5824 | N_acc: 0.2582
Epoch 004 | Loss: 2.7262 | T_acc: 0.5824 | N_acc: 0.2582
Epoch 005 | Loss: 2.7086 | T_acc: 0.5824 | N_acc: 0.2582
Epoch 006 | Loss: 2.6886 | T_acc: 0.5824 | N_acc: 0.2582
Epoch 007 | Loss: 2.6676 | T_acc: 0.5824 | N_acc: 0.2582
Epoch 008 | Loss: 2.6417 | T_acc: 0.5824 | N_acc: 0.2582
Epoch 009 | Loss: 2.6060 | T_acc: 0.5824 | N_acc: 0.2582
Epoch 010 | Loss: 2.5543 | T_acc: 0.5824 | N_acc: 0.3956
Epoch 011 | Loss: 2.4547 | T_acc: 0.5824 | N_acc: 0.4835
Epoch 012 | Loss: 2.3456 | T_acc: 0.5824 | N_acc: 0.4835
Epoch 013 | Loss: 2.3118 | T_acc: 0.5824 | N_acc: 0.4780
Epoch 014 | Loss: 2.2984 | T_acc: 0.5824 | N_acc: 0.4835
Epoch 015 | Loss: 2.2870 | T_acc: 0.5824 | N_acc: 0.4835
Epoch 016 | Loss: 2.2815 | T_acc: 0.5824 | N_acc: 0.4835
Epoch 017 | Loss: 2.2764 | T_acc: 0.5824 | N_acc: 0.4835
Epoch 018 | Loss: 2.2594 | T_ac

/var/folders/ff/flzt5myn3x55cy_nz85y_ynr0000gn/T/ipykernel_18667/2434738068.py:114: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/var/folders/ff/flzt5myn3x55cy_nz85y_ynr0000gn/T/ipykernel_18667/2434738068.py:135: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


Epoch 019 | Loss: 2.2118 | T_acc: 0.5769 | N_acc: 0.4835
Epoch 020 | Loss: 2.2250 | T_acc: 0.5824 | N_acc: 0.4835
Epoch 021 | Loss: 2.2296 | T_acc: 0.5824 | N_acc: 0.4835
Epoch 022 | Loss: 2.1842 | T_acc: 0.5879 | N_acc: 0.5055
Epoch 023 | Loss: 2.2116 | T_acc: 0.5879 | N_acc: 0.4835
Epoch 024 | Loss: 2.1613 | T_acc: 0.5989 | N_acc: 0.4890
Epoch 025 | Loss: 2.1792 | T_acc: 0.6044 | N_acc: 0.5055
Epoch 026 | Loss: 2.1769 | T_acc: 0.5934 | N_acc: 0.4945
Epoch 027 | Loss: 2.1790 | T_acc: 0.5769 | N_acc: 0.4945
Epoch 028 | Loss: 2.1439 | T_acc: 0.5879 | N_acc: 0.4890
Epoch 029 | Loss: 2.1258 | T_acc: 0.5989 | N_acc: 0.4945
Epoch 030 | Loss: 2.1045 | T_acc: 0.6154 | N_acc: 0.4890
Epoch 031 | Loss: 2.1224 | T_acc: 0.6099 | N_acc: 0.5110
Epoch 032 | Loss: 2.1272 | T_acc: 0.6044 | N_acc: 0.5000
Epoch 033 | Loss: 2.1039 | T_acc: 0.6044 | N_acc: 0.4780
Epoch 034 | Loss: 2.0864 | T_acc: 0.6154 | N_acc: 0.5110
Epoch 035 | Loss: 2.0789 | T_acc: 0.6538 | N_acc: 0.4945
Epoch 036 | Loss: 2.1012 | T_ac

...

Epoch 023 | Loss: 2.2217 | T_acc: 0.5824 | N_acc: 0.4835
Epoch 024 | Loss: 2.1994 | T_acc: 0.5824 | N_acc: 0.4835
Epoch 025 | Loss: 2.2060 | T_acc: 0.5824 | N_acc: 0.4835
Epoch 026 | Loss: 2.1969 | T_acc: 0.5824 | N_acc: 0.4835
Epoch 027 | Loss: 2.1644 | T_acc: 0.5824 | N_acc: 0.4835
Epoch 028 | Loss: 2.1708 | T_acc: 0.5879 | N_acc: 0.4945
Epoch 029 | Loss: 2.2116 | T_acc: 0.5824 | N_acc: 0.4890
Epoch 030 | Loss: 2.1913 | T_acc: 0.5824 | N_acc: 0.4945
Epoch 031 | Loss: 2.1758 | T_acc: 0.5824 | N_acc: 0.4835
Epoch 032 | Loss: 2.1143 | T_acc: 0.5824 | N_acc: 0.5000
Epoch 033 | Loss: 2.1036 | T_acc: 0.5824 | N_acc: 0.5000
Epoch 034 | Loss: 2.0627 | T_acc: 0.5824 | N_acc: 0.5275
Epoch 035 | Loss: 2.0526 | T_acc: 0.5824 | N_acc: 0.5165
Epoch 036 | Loss: 2.0411 | T_acc: 0.5824 | N_acc: 0.5275
Epoch 037 | Loss: 2.0914 | T_acc: 0.5934 | N_acc: 0.5330
Epoch 038 | Loss: 2.0276 | T_acc: 0.5934 | N_acc: 0.5330
Epoch 039 | Loss: 1.9833 | T_acc: 0.5879 | N_acc: 0.5495
Epoch 040 | Loss: 1.9581 | T_ac

/var/folders/ff/flzt5myn3x55cy_nz85y_ynr0000gn/T/ipykernel_18667/2434738068.py:114: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/var/folders/ff/flzt5myn3x55cy_nz85y_ynr0000gn/T/ipykernel_18667/2434738068.py:135: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


Epoch 020 | Loss: 2.1668 | T_acc: 0.5824 | N_acc: 0.4670
Epoch 021 | Loss: 2.1568 | T_acc: 0.5824 | N_acc: 0.5220
Epoch 022 | Loss: 2.1678 | T_acc: 0.5824 | N_acc: 0.5055
Epoch 023 | Loss: 2.0987 | T_acc: 0.5824 | N_acc: 0.5220
Epoch 024 | Loss: 2.0898 | T_acc: 0.5824 | N_acc: 0.5440
Epoch 025 | Loss: 2.0855 | T_acc: 0.5824 | N_acc: 0.5385
Epoch 026 | Loss: 2.0719 | T_acc: 0.5879 | N_acc: 0.5385
Epoch 027 | Loss: 2.0866 | T_acc: 0.5824 | N_acc: 0.5275
Epoch 028 | Loss: 2.0279 | T_acc: 0.5714 | N_acc: 0.5165
Epoch 029 | Loss: 1.9494 | T_acc: 0.5989 | N_acc: 0.5769
Epoch 030 | Loss: 1.9634 | T_acc: 0.6044 | N_acc: 0.5549
Epoch 031 | Loss: 2.0303 | T_acc: 0.6209 | N_acc: 0.5495
Epoch 032 | Loss: 1.9132 | T_acc: 0.5989 | N_acc: 0.5659
Epoch 033 | Loss: 1.8842 | T_acc: 0.6429 | N_acc: 0.6044
Epoch 034 | Loss: 1.7848 | T_acc: 0.6429 | N_acc: 0.5714
Epoch 035 | Loss: 1.7363 | T_acc: 0.6703 | N_acc: 0.6374
Epoch 036 | Loss: 1.7982 | T_acc: 0.6868 | N_acc: 0.6044
Epoch 037 | Loss: 1.7735 | T_ac

/var/folders/ff/flzt5myn3x55cy_nz85y_ynr0000gn/T/ipykernel_18667/2434738068.py:114: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/var/folders/ff/flzt5myn3x55cy_nz85y_ynr0000gn/T/ipykernel_18667/2434738068.py:135: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


Epoch 010 | Loss: 2.2638 | T_acc: 0.5824 | N_acc: 0.4835
Epoch 011 | Loss: 2.2470 | T_acc: 0.5824 | N_acc: 0.4835
Epoch 012 | Loss: 2.2456 | T_acc: 0.5824 | N_acc: 0.4835
Epoch 013 | Loss: 2.2274 | T_acc: 0.5824 | N_acc: 0.4835
Epoch 014 | Loss: 2.2010 | T_acc: 0.5824 | N_acc: 0.4835
Epoch 015 | Loss: 2.2297 | T_acc: 0.5824 | N_acc: 0.4835
Epoch 016 | Loss: 2.1976 | T_acc: 0.5769 | N_acc: 0.4835
Epoch 017 | Loss: 2.1815 | T_acc: 0.5879 | N_acc: 0.4890
Epoch 018 | Loss: 2.1573 | T_acc: 0.5879 | N_acc: 0.4835
Epoch 019 | Loss: 2.1635 | T_acc: 0.5934 | N_acc: 0.4835
Epoch 020 | Loss: 2.1242 | T_acc: 0.5989 | N_acc: 0.5110
Epoch 021 | Loss: 2.1083 | T_acc: 0.5934 | N_acc: 0.4890
Epoch 022 | Loss: 2.0313 | T_acc: 0.6154 | N_acc: 0.5330
Epoch 023 | Loss: 2.0781 | T_acc: 0.6374 | N_acc: 0.5110
Epoch 024 | Loss: 1.9808 | T_acc: 0.6264 | N_acc: 0.5055
Epoch 025 | Loss: 1.9891 | T_acc: 0.6099 | N_acc: 0.4615
Epoch 026 | Loss: 1.9375 | T_acc: 0.6099 | N_acc: 0.5275
Epoch 027 | Loss: 1.8575 | T_ac